In [ ]:
import json
import sys

import pandas as pd
from pathlib import Path
from io import BytesIO


# Path setup to import from the project root
project_root = Path().resolve().parent.parent.parent
sys.path.insert(0, str(project_root))

from event_synchronization.with_dynamic_events.wyscout import WyscoutSyncDynamicEventsManager
from event_synchronization.events_utils.wyscout import WyscoutEvents
from skillcorner.client import SkillcornerClient

skc_client = SkillcornerClient(username='USERNAME', password='PASSWORD')

# Load data

In [ ]:
data_dir = 'path/to/your/skc_dynamic_event/and/wyscout/match/folder/'

In [ ]:
# load match data with SKC match_id
MATCH_ID = 0
wyscout_match_data = skc_client.get_match(MATCH_ID, params={'matching': 'wyscout'})

# load Skillcorner dynamic events
api_skc_events = skc_client.get_dynamic_events(
                match_id=MATCH_ID, params={'file_format': 'csv', 'ignore_dynamic_events_check': False}
            )
skc_events = pd.read_csv(BytesIO(api_skc_events))

# load Wyscout raw events
raw_wyscout_events_path = data_dir / 'raw_wyscout_events.json'
with raw_wyscout_events_path.open() as f:
    raw_wyscout_events = json.load(f)

# Run DynamicEvent Synchronization Process

In [ ]:
wyscout_events_manager = WyscoutEvents(raw_wyscout_events, wyscout_match_data)
wyscout_manager = WyscoutSyncDynamicEventsManager(skc_events, raw_wyscout_events, wyscout_events_manager)

# apply the synchronization process
skc_events_mapping, wyscout_events_mapping = wyscout_manager.run()